In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.1:8b")
from langgraph.checkpoint.memory import InMemorySaver



/Users/ryankhurana/Desktop/langgraph2/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [4]:
def generate_joke(state:JokeState)->JokeState:
    prompt=f"Generate a joke on the topic of {state['topic']}"
    response=llm.invoke(prompt)
    return {'joke':response}

In [5]:
def generate_explanation(state:JokeState)->JokeState:
    prompt=f"Explain the joke{state['joke']}"
    response=llm.invoke(prompt).content
    return{"explanation":response}


In [6]:
graph=StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node("generate_explanation",generate_explanation)


graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)
checkpointer=InMemorySaver()# to apply persistance we create a checkpointer and pass it to the compile method of the graph
#this tell langgraph to save intial state,intermediate states and final state to the checkpointer so that we can resume the workflow from any point in case of failure or if we want to continue the workflow at a later time



workflow=graph.compile(checkpointer=checkpointer)



In [7]:
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [8]:
config1={'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':'programming'},config=config1)

{'topic': 'programming',
 'joke': AIMessage(content="Here's one:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs.", additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-14T13:13:19.983895Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11851092500, 'load_duration': 5165373667, 'prompt_eval_count': 18, 'prompt_eval_duration': 5705830875, 'eval_count': 17, 'eval_duration': 954571833, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cec7a-ab0c-7972-b987-8fd3389d5e0b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 17, 'total_tokens': 35}),
 'explanation': 'A joke about programmers!\n\nThe joke is a play on words, using a common concept in programming to make a pun. Here\'s the breakdown:\n\n* "Why do programmers prefer dark mode?" is the setup, asking for a reason why programmers might prefer a dark-colored theme for their screens.\n* 

In [13]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'programming', 'joke': AIMessage(content="Here's one:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs.", additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-14T03:03:02.758188Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8552407083, 'load_duration': 4066774333, 'prompt_eval_count': 18, 'prompt_eval_duration': 3487183541, 'eval_count': 17, 'eval_duration': 988083414, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cea4b-fbe1-7721-91a0-5d44b18cd957-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 17, 'total_tokens': 35}), 'explanation': 'A joke about programmers!\n\nThe joke is a play on words, using a common phrase from programming (bugs, as in software bugs) and applying it to a different context (light attracting bugs, as in insects). The punchline is a clever pun, implying that programmers prefe

In [16]:
list(workflow.get_state_history(config1)  )

[StateSnapshot(values={'topic': 'programming', 'joke': AIMessage(content="Here's one:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs.", additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-14T03:03:02.758188Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8552407083, 'load_duration': 4066774333, 'prompt_eval_count': 18, 'prompt_eval_duration': 3487183541, 'eval_count': 17, 'eval_duration': 988083414, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cea4b-fbe1-7721-91a0-5d44b18cd957-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 17, 'total_tokens': 35}), 'explanation': 'A joke about programmers!\n\nThe joke is a play on words, using a common phrase from programming (bugs, as in software bugs) and applying it to a different context (light attracting bugs, as in insects). The punchline is a clever pun, implying that programmers pref

In [18]:
config2={'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'pizza'},config=config2)

{'topic': 'pizza',
 'joke': AIMessage(content="Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-14T03:23:37.553575Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2550623666, 'load_duration': 103195000, 'prompt_eval_count': 18, 'prompt_eval_duration': 988045125, 'eval_count': 27, 'eval_duration': 1448053208, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cea5e-ead6-7d43-9b0d-dcc775f3a9a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 27, 'total_tokens': 45}),
 'explanation': 'A play on words!\n\nThe joke is a pun on the word "crusty." In one sense, a pizza crust is literally a crusty (flaky, crunchy) part of the pizza. But the word "crusty" can also mean being in a bad mood or being irritable.\n\nSo, the joke is saying that the pizza is "feeling 

In [ ]:
list(workflow.get_state_history(config2) )

[StateSnapshot(values={'topic': 'pizza', 'joke': AIMessage(content="Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-14T03:23:37.553575Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2550623666, 'load_duration': 103195000, 'prompt_eval_count': 18, 'prompt_eval_duration': 988045125, 'eval_count': 27, 'eval_duration': 1448053208, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cea5e-ead6-7d43-9b0d-dcc775f3a9a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 27, 'total_tokens': 45}), 'explanation': 'A play on words!\n\nThe joke is a pun on the word "crusty." In one sense, a pizza crust is literally a crusty (flaky, crunchy) part of the pizza. But the word "crusty" can also mean being in a bad mood or being irritable.\n\nSo, the joke is saying that th

In [14]:
checkpoint_cfg = {
    "configurable": {
        "thread_id": "2",
        "checkpoint_id": "1f11f551-56a5-642e-8000-6528740dd5fe",
    }
}
workflow.get_state(checkpoint_cfg)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f11f551-56a5-642e-8000-6528740dd5fe'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())